#  Rò rỉ dữ liệu: cột `duration`

So sánh AUC khi **CÓ** và **KHÔNG CÓ** cột `duration` để thấy rõ vì sao phải bỏ nó.


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 42


## 1. Đọc dữ liệu

In [8]:
# Đường dẫn đúng đi qua thư mục "bank-additional"
df = pd.read_csv("data/bank/bank-full.csv", sep=";")
y = (df["y"] == "yes").astype(int)
print(df.shape, "| tỉ lệ yes:", round(y.mean(), 3))


(45211, 17) | tỉ lệ yes: 0.117


## 2. Hàm huấn luyện + đánh giá nhanh (dùng lại cho 2 phiên bản)

In [9]:
def train_eval(X, y, label):
    cat_cols = X.select_dtypes(include="object").columns.tolist()
    prep = ColumnTransformer(
        [("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)],
        remainder="passthrough")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

    model = Pipeline([
        ("prep", prep),
        ("clf", RandomForestClassifier(
            n_estimators=300, max_features="sqrt",
            class_weight="balanced_subsample",
            n_jobs=-1, random_state=RANDOM_STATE)),
    ])
    model.fit(X_train, y_train)
    auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    print(f"{label:25s} ROC-AUC = {auc:.3f}")
    return auc


## 3. Chạy 2 phiên bản: CÓ và KHÔNG CÓ `duration`

In [10]:
X_with_duration = df.drop(columns=["y"])
X_without_duration = df.drop(columns=["y", "duration"])

auc_with = train_eval(X_with_duration, y, "CÓ duration")
auc_without = train_eval(X_without_duration, y, "KHÔNG có duration")


CÓ duration               ROC-AUC = 0.929
KHÔNG có duration         ROC-AUC = 0.792


## 4. BÀI HỌC VỀ RÒ RỈ DỮ LIỆU

- `duration` là thời lượng cuộc gọi — chỉ biết được **sau khi** đã gọi xong,
  nên không thể dùng để quyết định gọi ai **trước**.
- Cuộc gọi càng dài thường vì khách đã đồng ý → cột này gián tiếp "lộ" nhãn.
- AUC khi giữ `duration` cao giả tạo (~0.94), không phản ánh khả năng dự đoán
  thật khi triển khai (lúc đó ta chưa hề gọi điện).
- Tài liệu UCI cũng khuyến cáo bỏ cột này để có mô hình dự đoán thực tế.
- ➡️ **Từ notebook `02_random_forest.ipynb` trở đi, chỉ dùng phiên bản KHÔNG có `duration`.**
